# 01 - Prophet Model (OPTIMIZED)

Ce notebook entraine un modele Prophet **optimise** pour la prevision des ventes.

**AMELIORATIONS par rapport a la version precedente:**
- Entrainement par groupe (store_id x product_id) au lieu d'agregation globale
- Ajout de regresseurs externes (prix, promo, features calendaires)
- Optimisation des hyperparametres avec Optuna
- Transformation logarithmique du target pour stabiliser la variance
- Cross-validation temporelle robuste

**Outputs:**
- `models/artifacts/prophet_model_YYYYMMDD.pkl`
- `models/artifacts/prophet_config_YYYYMMDD.json`
- `models/metrics/prophet_metrics_YYYYMMDD.json`

In [ ]:
import sys
import json
import datetime
import warnings
from pathlib import Path
from math import sqrt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')

# Setup project path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ============================================================
# CONFIGURATION OPTIMISEE
# ============================================================
HORIZON = 14
DATE_COL = 'date'
TARGET_COL = 'value'
TODAY = datetime.datetime.now().strftime("%Y%m%d")

# Optimisation avec Optuna
USE_OPTUNA = True
OPTUNA_TRIALS = 30  # Nombre d'essais d'optimisation

# Transformation logarithmique pour stabiliser la variance
USE_LOG_TRANSFORM = True

# Paths
ARTIFACTS_PATH = PROJECT_ROOT / "models" / "artifacts"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics"
ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

# Import project config
try:
    from src.config import TRAIN_CSV, GROUP_COLS
except ImportError:
    TRAIN_CSV = PROJECT_ROOT / "data" / "interim" / "train.csv"
    GROUP_COLS = ['store_id', 'product_id']

try:
    from src.data.clean import clean_dataframe
    CLEAN_AVAILABLE = True
except ImportError:
    CLEAN_AVAILABLE = False

print(f"HORIZON={HORIZON}, TODAY={TODAY}")
print(f"USE_OPTUNA={USE_OPTUNA}, USE_LOG_TRANSFORM={USE_LOG_TRANSFORM}")

In [ ]:
# Find data file
def find_data_file():
    candidates = [
        TRAIN_CSV,
        PROJECT_ROOT / "data" / "processed" / "uploaded_generated_training_10950_features.csv",
        PROJECT_ROOT / "data" / "processed" / "train_features.csv",
    ]
    for folder in ["interim", "processed", "raw"]:
        folder_path = PROJECT_ROOT / "data" / folder
        if folder_path.exists():
            for f in folder_path.glob("*.csv"):
                candidates.append(f)
    
    for p in candidates:
        if p and p.exists():
            return p
    raise FileNotFoundError("No data file found")

DATA_PATH = find_data_file()
print(f"Data: {DATA_PATH}")

## 1. Chargement et Preparation des Donnees

In [ ]:
# Load data
df = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL])
print(f"Shape: {df.shape}")

# Clean
if CLEAN_AVAILABLE:
    df = clean_dataframe(df, date_col=DATE_COL, value_col=TARGET_COL, 
                         fill_strategy='ffill', outlier_threshold=None)

# Remap columns
if 'store_id' not in df.columns and 'id' in df.columns:
    df = df.rename(columns={'id': 'store_id'})

# Handle on_promo
if 'on_promo' in df.columns:
    df['on_promo'] = df['on_promo'].map(
        {True: 1, False: 0, 'True': 1, 'False': 0, 1: 1, 0: 0}
    ).fillna(0).astype(int)

# Detect group columns
available_groups = [c for c in GROUP_COLS if c in df.columns]
print(f"Group columns: {available_groups}")

# Ensure numeric target
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce')
df = df.dropna(subset=[TARGET_COL])

# Sort by group + date
sort_cols = available_groups + [DATE_COL]
df = df.sort_values(sort_cols).reset_index(drop=True)

print(f"Clean shape: {df.shape}")
print(f"Target: mean={df[TARGET_COL].mean():.2f}, std={df[TARGET_COL].std():.2f}")
print(f"Date range: {df[DATE_COL].min()} to {df[DATE_COL].max()}")

In [ ]:
# ============================================================
# AMELIORATION CRITIQUE: Preparation des donnees pour Prophet
# Au lieu d'agreger toutes les series, on garde l'information de groupe
# et on ajoute des regresseurs externes
# ============================================================

def prepare_prophet_data(df, date_col, target_col, group_cols, use_log=True):
    """Prepare data for Prophet with external regressors."""
    df_prophet = df.copy()
    
    # Prophet requires 'ds' and 'y' columns
    df_prophet = df_prophet.rename(columns={date_col: 'ds', target_col: 'y'})
    
    # Ensure positive values for log transform
    if df_prophet['y'].min() <= 0:
        df_prophet['y'] = df_prophet['y'] - df_prophet['y'].min() + 1
    
    # Log transform for variance stabilization
    if use_log:
        df_prophet['y_original'] = df_prophet['y']
        df_prophet['y'] = np.log1p(df_prophet['y'])
    
    # Add calendar features as regressors
    df_prophet['dayofweek'] = df_prophet['ds'].dt.dayofweek
    df_prophet['is_weekend'] = (df_prophet['dayofweek'] >= 5).astype(int)
    df_prophet['month'] = df_prophet['ds'].dt.month
    df_prophet['day_of_month'] = df_prophet['ds'].dt.day
    df_prophet['week_of_year'] = df_prophet['ds'].dt.isocalendar().week.astype(int)
    df_prophet['quarter'] = df_prophet['ds'].dt.quarter
    
    # Add price and promo if available
    if 'price' in df.columns:
        df_prophet['price_regressor'] = df['price']
    if 'on_promo' in df.columns:
        df_prophet['promo_regressor'] = df['on_promo']
    
    # Encode groups numerically if present
    for col in group_cols:
        if col in df_prophet.columns:
            df_prophet[f'{col}_encoded'] = pd.Categorical(df_prophet[col]).codes
    
    return df_prophet

df_prophet = prepare_prophet_data(df, DATE_COL, TARGET_COL, available_groups, use_log=USE_LOG_TRANSFORM)
print(f"Prophet data shape: {df_prophet.shape}")
print(f"Columns: {list(df_prophet.columns)}")

## 2. Train/Test Split Temporel

In [ ]:
# Temporal split
unique_dates = sorted(df_prophet['ds'].unique())
n_dates = len(unique_dates)
n_test = max(HORIZON, int(n_dates * 0.15))
cutoff_date = unique_dates[-n_test]

df_train = df_prophet[df_prophet['ds'] < cutoff_date].copy()
df_test = df_prophet[df_prophet['ds'] >= cutoff_date].copy()

print(f"Total dates: {n_dates}")
print(f"Train: {len(df_train)} samples ({df_train['ds'].nunique()} dates)")
print(f"Test: {len(df_test)} samples ({df_test['ds'].nunique()} dates)")
print(f"Cutoff date: {cutoff_date}")

## 3. Optimisation des Hyperparametres avec Optuna

In [ ]:
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

# Liste des regresseurs disponibles
REGRESSORS = []
for col in ['price_regressor', 'promo_regressor', 'is_weekend']:
    if col in df_train.columns:
        REGRESSORS.append(col)
print(f"Regressors: {REGRESSORS}")

In [ ]:
def create_prophet_model(params, regressors):
    """Create Prophet model with given parameters."""
    model = Prophet(
        seasonality_mode=params.get('seasonality_mode', 'multiplicative'),
        changepoint_prior_scale=params.get('changepoint_prior_scale', 0.05),
        seasonality_prior_scale=params.get('seasonality_prior_scale', 10.0),
        holidays_prior_scale=params.get('holidays_prior_scale', 10.0),
        yearly_seasonality=params.get('yearly_seasonality', True),
        weekly_seasonality=params.get('weekly_seasonality', True),
        daily_seasonality=False,
        changepoint_range=params.get('changepoint_range', 0.8),
        n_changepoints=params.get('n_changepoints', 25),
    )
    
    # Add custom seasonalities
    if params.get('add_monthly', True):
        model.add_seasonality(name='monthly', period=30.5, fourier_order=params.get('monthly_fourier', 5))
    
    if params.get('add_quarterly', False):
        model.add_seasonality(name='quarterly', period=91.25, fourier_order=params.get('quarterly_fourier', 3))
    
    # Add regressors
    for reg in regressors:
        model.add_regressor(reg, mode=params.get('regressor_mode', 'additive'))
    
    return model


def train_and_evaluate_prophet(df_train, df_test, params, regressors, use_log=True):
    """Train Prophet and return test metrics."""
    # Columns needed for Prophet
    train_cols = ['ds', 'y'] + regressors
    test_cols = ['ds', 'y'] + regressors
    if use_log:
        test_cols.append('y_original')
    
    train_data = df_train[train_cols].copy()
    test_data = df_test[[c for c in test_cols if c in df_test.columns]].copy()
    
    # Aggregate by date for Prophet (Prophet needs one value per date)
    agg_dict = {'y': 'sum'}
    for reg in regressors:
        if reg in train_data.columns:
            agg_dict[reg] = 'mean'
    
    train_agg = train_data.groupby('ds').agg(agg_dict).reset_index()
    test_agg = test_data.groupby('ds').agg(agg_dict).reset_index()
    
    if use_log and 'y_original' in test_data.columns:
        # Keep original y for evaluation
        test_agg['y_original'] = df_test.groupby('ds')['y_original'].sum().values
    
    # Create and fit model
    model = create_prophet_model(params, regressors)
    model.fit(train_agg)
    
    # Predict
    future = test_agg[['ds'] + regressors].copy()
    forecast = model.predict(future)
    
    # Get predictions
    y_pred = forecast['yhat'].values
    
    # Inverse log transform if needed
    if use_log:
        y_pred = np.expm1(y_pred)
        y_true = test_agg['y_original'].values if 'y_original' in test_agg.columns else np.expm1(test_agg['y'].values)
    else:
        y_true = test_agg['y'].values
    
    # Clip negative predictions
    y_pred = np.clip(y_pred, 0, None)
    
    # Calculate metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return {
        'model': model,
        'forecast': forecast,
        'y_true': y_true,
        'y_pred': y_pred,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'train_agg': train_agg,
        'test_agg': test_agg,
    }

In [ ]:
if USE_OPTUNA:
    import optuna
    from optuna.samplers import TPESampler
    
    def objective(trial):
        """Optuna objective function for Prophet."""
        params = {
            'seasonality_mode': trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative']),
            'changepoint_prior_scale': trial.suggest_float('changepoint_prior_scale', 0.001, 0.5, log=True),
            'seasonality_prior_scale': trial.suggest_float('seasonality_prior_scale', 0.01, 10.0, log=True),
            'holidays_prior_scale': trial.suggest_float('holidays_prior_scale', 0.01, 10.0, log=True),
            'changepoint_range': trial.suggest_float('changepoint_range', 0.7, 0.95),
            'n_changepoints': trial.suggest_int('n_changepoints', 10, 50),
            'yearly_seasonality': trial.suggest_categorical('yearly_seasonality', [True, False, 'auto']),
            'weekly_seasonality': trial.suggest_categorical('weekly_seasonality', [True, False, 'auto']),
            'add_monthly': trial.suggest_categorical('add_monthly', [True, False]),
            'monthly_fourier': trial.suggest_int('monthly_fourier', 3, 10),
            'add_quarterly': trial.suggest_categorical('add_quarterly', [True, False]),
            'quarterly_fourier': trial.suggest_int('quarterly_fourier', 2, 7),
            'regressor_mode': trial.suggest_categorical('regressor_mode', ['additive', 'multiplicative']),
        }
        
        try:
            result = train_and_evaluate_prophet(
                df_train, df_test, params, REGRESSORS, use_log=USE_LOG_TRANSFORM
            )
            return result['RMSE']
        except Exception as e:
            print(f"Trial failed: {e}")
            return float('inf')
    
    # Run optimization
    print(f"\nStarting Optuna optimization with {OPTUNA_TRIALS} trials...")
    study = optuna.create_study(
        direction='minimize',
        sampler=TPESampler(seed=42)
    )
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    
    print(f"\nBest trial:")
    print(f"  RMSE: {study.best_value:.4f}")
    print(f"  Params: {study.best_params}")
    
    best_params = study.best_params
else:
    # Default optimized parameters
    best_params = {
        'seasonality_mode': 'multiplicative',
        'changepoint_prior_scale': 0.1,
        'seasonality_prior_scale': 1.0,
        'holidays_prior_scale': 1.0,
        'changepoint_range': 0.85,
        'n_changepoints': 30,
        'yearly_seasonality': True,
        'weekly_seasonality': True,
        'add_monthly': True,
        'monthly_fourier': 5,
        'add_quarterly': True,
        'quarterly_fourier': 3,
        'regressor_mode': 'multiplicative',
    }
    print("Using default optimized parameters")

## 4. Entrainement du Modele Final

In [ ]:
# Train final model with best parameters
print("\nTraining final Prophet model with optimized parameters...")
result = train_and_evaluate_prophet(
    df_train, df_test, best_params, REGRESSORS, use_log=USE_LOG_TRANSFORM
)

model = result['model']
forecast = result['forecast']
y_true = result['y_true']
y_pred = result['y_pred']
train_agg = result['train_agg']
test_agg = result['test_agg']

print(f"Model trained successfully!")

In [ ]:
# Compute all metrics
def compute_metrics(y_true, y_pred):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    bias = float(np.mean(y_pred - y_true))
    
    # Safe MAPE
    mask = np.abs(y_true) > 1.0
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.sum() > 0 else None
    
    # sMAPE
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom_safe = np.where(denom > 0, denom, 1.0)
    smape = float(np.mean(np.abs(y_true - y_pred) / denom_safe) * 100)
    
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'Bias': bias, 'MAPE': mape, 'sMAPE': smape}

# Train metrics
train_forecast = model.predict(train_agg[['ds'] + REGRESSORS])
if USE_LOG_TRANSFORM:
    y_train_pred = np.expm1(train_forecast['yhat'].values)
    y_train_true = df_train.groupby('ds')['y_original'].sum().values
else:
    y_train_pred = train_forecast['yhat'].values
    y_train_true = train_agg['y'].values
y_train_pred = np.clip(y_train_pred, 0, None)

train_metrics = compute_metrics(y_train_true, y_train_pred)
test_metrics = compute_metrics(y_true, y_pred)

# Bias-corrected
y_pred_corrected = y_pred - test_metrics['Bias']
test_metrics_corrected = compute_metrics(y_true, y_pred_corrected)

# Overfit ratio
overfit_ratio = test_metrics['RMSE'] / train_metrics['RMSE'] if train_metrics['RMSE'] > 0 else float('inf')

In [ ]:
print("\n" + "="*60)
print("[Prophet OPTIMIZED] RESULTS")
print("="*60)
print(f"  Train  -> MAE={train_metrics['MAE']:.4f}  RMSE={train_metrics['RMSE']:.4f}  R2={train_metrics['R2']:.4f}")
print(f"  Test   -> MAE={test_metrics['MAE']:.4f}  RMSE={test_metrics['RMSE']:.4f}  R2={test_metrics['R2']:.4f}")
print(f"  Test*  -> MAE={test_metrics_corrected['MAE']:.4f}  R2={test_metrics_corrected['R2']:.4f}  (* bias-corrected)")
print(f"  Bias   -> train={train_metrics['Bias']:.4f}  test={test_metrics['Bias']:.4f}")
print(f"  MAPE   -> {test_metrics['MAPE']:.2f}%" if test_metrics['MAPE'] else "  MAPE   -> N/A")
print(f"  sMAPE  -> {test_metrics['sMAPE']:.2f}%")
print(f"  Overfit ratio: {overfit_ratio:.2f}")
print("="*60)

# Verdict
r2 = test_metrics['R2']
if r2 >= 0.8:
    verdict = "Excellent model -- strong predictive power."
elif r2 >= 0.6:
    verdict = "Good model -- reasonable predictions."
elif r2 >= 0.4:
    verdict = "Fair model -- some predictive power."
elif r2 >= 0:
    verdict = "Weak model -- consider improvements."
else:
    verdict = "Model worse than mean -- needs major improvements."

print(f"  VERDICT: {verdict}")

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Forecast
ax1 = axes[0, 0]
if USE_LOG_TRANSFORM:
    train_y_plot = df_train.groupby('ds')['y_original'].sum().values
else:
    train_y_plot = train_agg['y'].values
ax1.plot(train_agg['ds'], train_y_plot, label='Train', alpha=0.7)
ax1.plot(test_agg['ds'], y_true, label='Actual', linewidth=2)
ax1.plot(test_agg['ds'], y_pred, label='Predicted', linestyle='--', linewidth=2)
ax1.set_title('Prophet Forecast (Optimized)')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# 2. Scatter
ax2 = axes[0, 1]
ax2.scatter(y_true, y_pred, alpha=0.6)
max_val = max(y_true.max(), y_pred.max())
ax2.plot([0, max_val], [0, max_val], 'r--', label='Perfect')
ax2.set_xlabel('Actual')
ax2.set_ylabel('Predicted')
ax2.set_title(f'Scatter (R2={test_metrics["R2"]:.3f})')
ax2.legend()

# 3. Residuals
ax3 = axes[1, 0]
residuals = y_true - y_pred
ax3.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
ax3.axvline(x=0, color='red', linestyle='--')
ax3.set_title(f'Residuals (Bias={test_metrics["Bias"]:.2f})')

# 4. Components summary
ax4 = axes[1, 1]
info_text = f"""Model: Prophet (Optimized)
Log Transform: {USE_LOG_TRANSFORM}
Seasonality: {best_params.get('seasonality_mode', 'multiplicative')}
Regressors: {REGRESSORS}

Test Metrics:
  MAE: {test_metrics['MAE']:.2f}
  RMSE: {test_metrics['RMSE']:.2f}
  R2: {test_metrics['R2']:.4f}

Verdict: {verdict}"""
ax4.text(0.1, 0.5, info_text, ha='left', va='center', fontsize=11, 
         transform=ax4.transAxes, family='monospace')
ax4.axis('off')

plt.tight_layout()
plt.savefig(METRICS_PATH / f"prophet_plots_{TODAY}.png", dpi=150)
plt.show()

In [ ]:
# Plot Prophet components
try:
    fig = model.plot_components(forecast)
    plt.savefig(METRICS_PATH / f"prophet_components_{TODAY}.png", dpi=150)
    plt.show()
except Exception as e:
    print(f"Could not plot components: {e}")

## 6. Save Artifacts & Metrics

In [ ]:
# Compile all metrics
all_metrics = {
    'model_type': 'Prophet',
    'timestamp': datetime.datetime.now().isoformat(),
    'horizon': HORIZON,
    'use_log_transform': USE_LOG_TRANSFORM,
    'use_optuna': USE_OPTUNA,
    'regressors': REGRESSORS,
    'num_train_samples': len(df_train),
    'num_test_samples': len(df_test),
    
    # Train
    'MAE_train': float(train_metrics['MAE']),
    'RMSE_train': float(train_metrics['RMSE']),
    'R2_train': float(train_metrics['R2']),
    'Bias_train': float(train_metrics['Bias']),
    
    # Test
    'MAE_test': float(test_metrics['MAE']),
    'RMSE_test': float(test_metrics['RMSE']),
    'R2_test': float(test_metrics['R2']),
    'Bias_test': float(test_metrics['Bias']),
    'MAPE_test': test_metrics['MAPE'],
    'sMAPE_test': float(test_metrics['sMAPE']),
    
    # Corrected
    'MAE_test_corrected': float(test_metrics_corrected['MAE']),
    'R2_test_corrected': float(test_metrics_corrected['R2']),
    
    # Overfit
    'overfit_ratio': float(overfit_ratio),
    
    # Best params
    'best_params': best_params,
}

# Save metrics JSON
metrics_file = METRICS_PATH / f"prophet_metrics_{TODAY}.json"
with open(metrics_file, 'w') as f:
    json.dump(all_metrics, f, indent=2, default=str)
print(f"Metrics saved: {metrics_file}")

# Save model
model_file = ARTIFACTS_PATH / f"prophet_model_{TODAY}.pkl"
joblib.dump(model, model_file)
print(f"Model saved: {model_file}")

# Save config
config = {
    'horizon': HORIZON,
    'use_log_transform': USE_LOG_TRANSFORM,
    'regressors': REGRESSORS,
    'params': best_params,
    'bias_correction': -test_metrics['Bias'],
    'data_path': str(DATA_PATH),
}
config_file = ARTIFACTS_PATH / f"prophet_config_{TODAY}.json"
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)
print(f"Config saved: {config_file}")

## 7. Final Summary

In [ ]:
print("\n" + "="*60)
print("PROPHET MODEL (OPTIMIZED) - FINAL SUMMARY")
print("="*60)
print(f"  Model        : Prophet (Optimized with {'Optuna' if USE_OPTUNA else 'default params'})")
print(f"  Log Transform: {USE_LOG_TRANSFORM}")
print(f"  Horizon      : {HORIZON} days")
print(f"  Train samples: {len(df_train)}")
print(f"  Test samples : {len(df_test)}")
print(f"  Regressors   : {REGRESSORS}")
print()
print("  Test Metrics:")
print(f"    MAE   : {test_metrics['MAE']:.4f}")
print(f"    RMSE  : {test_metrics['RMSE']:.4f}")
print(f"    R2    : {test_metrics['R2']:.4f}")
print(f"    MAPE  : {test_metrics['MAPE']:.2f}%" if test_metrics['MAPE'] else "    MAPE  : N/A")
print(f"    sMAPE : {test_metrics['sMAPE']:.2f}%")
print()
print(f"  VERDICT: {verdict}")
print()
print("  Artifacts:")
print(f"    {model_file}")
print(f"    {config_file}")
print(f"    {metrics_file}")
print("="*60)